In [1]:
"""
Week 6 — Feature Engineering (bed/bath ratio, property age, school district)

What this notebook does, in order:
  1. Re-derives the Week 3 cleaned dataset FROM RAW, this time keeping
     Latitude/Longitude (Week 3 dropped them — they are needed here for the
     spatial join and were never saved to cleaned_single_family_sales_
     preprocessed.csv).
  2. Adds two new logical-impossibility cleaning checks that only apply once
     Latitude/Longitude and a spatial join enter the picture (Sec. 02):
     missing coordinates, and YearBuilt after the sale's close year.
  3. Engineers BedBathRatio and PropertyAge.
  4. Spatially joins each property against CA Unified School District
     boundaries (GeoJSON) to add a DistrictName feature.
  5. Saves ONE enriched dataset (cleaned_single_family_sales_engineered.csv)
     that is used for every model run below, and is intended to replace
     cleaned_single_family_sales_preprocessed.csv going forward.
  6. Re-trains Linear Regression / Decision Tree / Random Forest under three
     feature-set configurations on that SAME enriched dataset (same rows),
     so any R2 difference reflects the feature set, not a different row
     count from re-cleaning:
       - "Old (Week 5)"            — exact Week 5 feature set
       - "New: engineered numeric" — old + BedBathRatio + PropertyAge
       - "New: + school district"  — old + engineered numeric + DistrictName
         (target-encoded the same CV-safe way as City)

Key assumptions / limitations (Sec. 10 — called out explicitly, not left
implicit in code):
  - Assumes the GeoJSON is in (or declares) a CRS; points are built in
    EPSG:4326 (lat/long) and reprojected to match the district file's CRS if
    it differs.
  - A property whose coordinates fall inside a district boundary that is
    NOT of DistrictType "Unified" (e.g. it's only covered by separate
    elementary/high school districts) will get a missing DistrictName by
    design (Sec. 06: this is treated as informative missingness, not an
    error, and flows through the same imputer/missing-indicator machinery
    as any other missing categorical).
  - Properties that intersect more than one Unified polygon (edge cases at
    shared boundaries, sliver overlaps) are logged and resolved by keeping
    the first match, so the row count never inflates from the join.
"""

'\nWeek 6 — Feature Engineering (bed/bath ratio, property age, school district)\n\nWhat this notebook does, in order:\n  1. Re-derives the Week 3 cleaned dataset FROM RAW, this time keeping\n     Latitude/Longitude (Week 3 dropped them — they are needed here for the\n     spatial join and were never saved to cleaned_single_family_sales_\n     preprocessed.csv).\n  2. Adds two new logical-impossibility cleaning checks that only apply once\n     Latitude/Longitude and a spatial join enter the picture (Sec. 02):\n     missing coordinates, and YearBuilt after the sale\'s close year.\n  3. Engineers BedBathRatio and PropertyAge.\n  4. Spatially joins each property against CA Unified School District\n     boundaries (GeoJSON) to add a DistrictName feature.\n  5. Saves ONE enriched dataset (cleaned_single_family_sales_engineered.csv)\n     that is used for every model run below, and is intended to replace\n     cleaned_single_family_sales_preprocessed.csv going forward.\n  6. Re-trains Linear

In [2]:
import numpy as np
import pandas as pd
import geopandas as gpd
import warnings
from shapely.geometry import Point
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.model_selection import KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeRegressor

In [3]:
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

In [4]:
RAW_PATH = "cleaned_single_family_sales.csv"       
SCHOOL_DISTRICT_PATH = "CA_school_district_areas.geojson"

In [5]:
# ---------------------------------------------------------------------------
# Load raw data and select columns (same as 02_preprocessing.py, PLUS the
# coordinate columns needed for the spatial join).
# ---------------------------------------------------------------------------
keep_cols = [
    "CloseDate",
    "ClosePrice",              # target

    # numeric
    "LivingArea",
    "BedroomsTotal",
    "BathroomsTotalInteger",
    "LotSizeSquareFeet",
    "GarageSpaces",
    "YearBuilt",
    "ParkingTotal",
    "Latitude",
    "Longitude",

    # categorical
    "PropertySubType",
    "City",
    "Levels",
    "AttachedGarageYN",
    "PoolPrivateYN",
    "FireplaceYN",
    "NewConstructionYN",
]

In [6]:
df = pd.read_csv(RAW_PATH, low_memory=False)
print(f"Original shape: {df.shape}")
df = df[keep_cols].copy()

Original shape: (333364, 83)


In [7]:
# ---------------------------------------------------------------------------
# Normalize categorical dtypes (identical rationale to 02_preprocessing.py:
# YN fields can arrive as a mix of real bools and "Y"/"N" strings, which
# breaks OneHotEncoder later).
# ---------------------------------------------------------------------------
cat_cols_all = [
    "PropertySubType", "City", "Levels",
    "AttachedGarageYN", "PoolPrivateYN", "FireplaceYN", "NewConstructionYN",
]
for col in cat_cols_all:
    df[col] = df[col].where(df[col].isna(), df[col].astype(str))

In [8]:
df = df.rename(columns={
    "BedroomsTotal": "Bedrooms",
    "BathroomsTotalInteger": "Bathrooms",
    "LotSizeSquareFeet": "LotSize",
})

In [9]:
numeric_cols = [
    "ClosePrice", "LivingArea", "Bedrooms", "Bathrooms",
    "LotSize", "GarageSpaces", "YearBuilt", "ParkingTotal",
    "Latitude", "Longitude",
]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

In [10]:
df["CloseDate"] = pd.to_datetime(df["CloseDate"], format="mixed", errors="coerce")

In [11]:
def log_drop(df_before, df_after, reason):
    """Sec. 02: log the count/percentage of rows removed at each cleaning
    step so an undocumented drop is easy for a reviewer to catch."""
    n_dropped = len(df_before) - len(df_after)
    pct = 100 * n_dropped / len(df_before) if len(df_before) else 0.0
    print(f"  Dropped {n_dropped} rows ({pct:.2f}%) — {reason}")
    return df_after

In [12]:
# ---------------------------------------------------------------------------
# Row-level cleaning — identical to 02_preprocessing.py, plus two new checks
# that only matter once coordinates enter the pipeline.
# ---------------------------------------------------------------------------
n0 = len(df)
print(f"\nStarting row-level cleaning from {n0} rows")


Starting row-level cleaning from 333364 rows


In [13]:
_before = df
df = df[df["ClosePrice"] > 0]
df = log_drop(_before, df, "ClosePrice <= 0")

  Dropped 3 rows (0.00%) — ClosePrice <= 0


In [14]:
_before = df
df = df.dropna(subset=["ClosePrice", "CloseDate"])
df = log_drop(_before, df, "missing target or CloseDate")

  Dropped 0 rows (0.00%) — missing target or CloseDate


In [15]:
_before = df
df = df.drop_duplicates()
df = log_drop(_before, df, "exact duplicate rows")

  Dropped 68 rows (0.02%) — exact duplicate rows


In [16]:
_before = df
df = df[df["LivingArea"].isna() | (df["LivingArea"] > 0)]
df = log_drop(_before, df, "LivingArea <= 0 (logical impossibility)")

  Dropped 135 rows (0.04%) — LivingArea <= 0 (logical impossibility)


In [17]:
_before = df
df = df[df["Bedrooms"].isna() | (df["Bedrooms"] >= 0)]
df = df[df["Bathrooms"].isna() | (df["Bathrooms"] >= 0)]
df = log_drop(_before, df, "negative Bedrooms/Bathrooms")

  Dropped 0 rows (0.00%) — negative Bedrooms/Bathrooms


In [18]:
# NEW: coordinates are required for the spatial join, so rows missing either
# one cannot get a DistrictName under any circumstance — dropped rather than
# silently left null, and logged per Sec. 02.
_before = df
df = df.dropna(subset=["Latitude", "Longitude"])
df = log_drop(_before, df, "missing Latitude/Longitude")

  Dropped 102 rows (0.03%) — missing Latitude/Longitude


In [19]:
# NEW logical impossibility (Sec. 02): a home cannot be built after the year
# it was sold.
_before = df
df = df[df["YearBuilt"].isna() | (df["YearBuilt"] <= df["CloseDate"].dt.year)]
df = log_drop(_before, df, "YearBuilt after CloseDate year (logical impossibility)")

  Dropped 28 rows (0.01%) — YearBuilt after CloseDate year (logical impossibility)


In [20]:
print(f"Rows remaining after cleaning: {len(df)} ({100 * len(df) / n0:.2f}% of original)\n")

Rows remaining after cleaning: 333028 (99.90% of original)



In [21]:
# ---------------------------------------------------------------------------
# Missingness indicators (Sec. 06), added BEFORE imputation/engineering so
# the flag itself can be predictive. Target is never flagged.
# ---------------------------------------------------------------------------
BASE_NUMERIC_BASE_NAMES = [
    "LivingArea", "Bedrooms", "Bathrooms", "LotSize",
    "GarageSpaces", "YearBuilt", "ParkingTotal",
]
for col in BASE_NUMERIC_BASE_NAMES:
    if df[col].isna().any():
        df[f"{col}_missing"] = df[col].isna().astype(int)

In [22]:
# ---------------------------------------------------------------------------
# Feature engineering: bed/bath ratio, property age
# ---------------------------------------------------------------------------
# BedBathRatio: undefined for 0-bathroom homes (division by zero) — treated
# as missing rather than inf, since a 0-bath listing is itself unusual
# (likely a data issue or a bare-land/non-standard listing) and the
# missing-indicator flag lets the model learn that pattern directly (Sec. 06)
# instead of being handed a nonsensical inf/large value.
df["BedBathRatio"] = df["Bedrooms"] / df["Bathrooms"].replace(0, np.nan)

In [23]:
# PropertyAge: age in years at the time of sale, not just YearBuilt, so the
# feature reflects condition-relevant aging (Best Practices Sec. 05).
df["PropertyAge"] = df["CloseDate"].dt.year - df["YearBuilt"]

In [24]:
ENGINEERED_BASE_NAMES = ["BedBathRatio", "PropertyAge"]
for col in ENGINEERED_BASE_NAMES:
    if df[col].isna().any():
        df[f"{col}_missing"] = df[col].isna().astype(int)

In [25]:
print("Engineered BedBathRatio and PropertyAge.")
print(df[["BedBathRatio", "PropertyAge"]].describe())

Engineered BedBathRatio and PropertyAge.
        BedBathRatio    PropertyAge
count  332874.000000  332815.000000
mean        1.459082      49.124547
std         0.476485      27.492226
min         0.000000       0.000000
25%         1.000000      27.000000
50%         1.500000      49.000000
75%         1.500000      69.000000
max         7.500000     249.000000


In [26]:
# ---------------------------------------------------------------------------
# School district spatial join
# ---------------------------------------------------------------------------
districts = gpd.read_file(SCHOOL_DISTRICT_PATH)
districts = districts.set_crs("EPSG: 3857", allow_override=True)
print(f"Districts CRS (explicitly set): {districts.crs}")
print(f"Loaded {len(districts)} district polygons; columns: {list(districts.columns)}")

Districts CRS (explicitly set): EPSG: 3857
Loaded 936 district polygons; columns: ['OBJECTID', 'Year', 'FedID', 'CDCode', 'CDSCode', 'CountyName', 'DistrictName', 'DistrictType', 'GradeLow', 'GradeHigh', 'GradeLowCensus', 'GradeHighCensus', 'AssistStatus', 'UpdateNotes', 'EnrollTotal', 'EnrollCharter', 'EnrollNonCharter', 'AAcount', 'AApct', 'AIcount', 'AIpct', 'AScount', 'ASpct', 'FIcount', 'FIpct', 'HIcount', 'HIpct', 'PIcount', 'PIpct', 'WHcount', 'WHpct', 'MRcount', 'MRpct', 'NRcount', 'NRpct', 'ELcount', 'ELpct', 'FOScount', 'FOSpct', 'HOMcount', 'HOMpct', 'MIGcount', 'MIGpct', 'SWDcount', 'SWDpct', 'SEDcount', 'SEDpct', 'DistrctAreaSqMi', 'LocaleCode', 'LocaleDesc', 'geometry']


In [27]:
# Step 3: keep only Unified school districts, per the assignment spec.
n_before_filter = len(districts)
districts = districts[districts["DistrictType"] == "Unified"].copy()
print(f"Filtered to DistrictType == 'Unified': {len(districts)} / {n_before_filter} kept")

Filtered to DistrictType == 'Unified': 345 / 936 kept


In [28]:
# Step 4: convert each property's Latitude/Longitude into a geographic point.
# Property coordinates come from the MLS extract as plain lat/long degrees,
# so points are built in WGS84 (EPSG:4326) regardless of what CRS the
# district file uses.
points = gpd.GeoDataFrame(
    df.copy(),
    geometry=gpd.points_from_xy(df["Longitude"], df["Latitude"]),
    crs="EPSG:4326",
)

In [29]:
# Reproject the points into the district file's CRS (EPSG:3857) so the
# spatial join is comparing coordinates on the same footing. This is now an
# unconditional reprojection rather than an "if the CRS differs" check,
# since we've confirmed from the file itself that it's always going to
# differ (4326 degrees vs. 3857 meters).
points = points.to_crs(districts.crs)
print(f"Points CRS (reprojected): {points.crs}   Districts CRS: {districts.crs}")

Points CRS (reprojected): EPSG: 3857   Districts CRS: EPSG: 3857


In [30]:
# Step 5: spatial join — which Unified polygon contains each property.
# how="left" keeps every property row even if no Unified polygon contains it
# (expected for areas covered only by separate elementary/high districts).
joined = gpd.sjoin(
    points,
    districts[["DistrictName", "geometry"]],
    how="left",
    predicate="within",
)

In [31]:
# A point exactly on a shared boundary, or a sliver polygon overlap, can
# match more than one Unified polygon and duplicate that row. Keep the first
# match and log how often this happened, so the row count never inflates.
n_before_dedup = len(joined)
joined = joined[~joined.index.duplicated(keep="first")]
n_dupe = n_before_dedup - len(joined)
print(f"Rows with multiple Unified-district matches (kept first): {n_dupe}")

Rows with multiple Unified-district matches (kept first): 0


In [32]:
n_unmatched = joined["DistrictName"].isna().sum()
print(f"Properties with no Unified-district match: {n_unmatched} "
      f"({100 * n_unmatched / len(joined):.2f}%) — left as missing, not dropped")

Properties with no Unified-district match: 83283 (25.01%) — left as missing, not dropped


In [33]:
# Step 6: attach DistrictName back onto the working dataframe.
df["DistrictName"] = joined["DistrictName"].values
df = df.drop(columns=["geometry"], errors="ignore")

In [34]:
print(df["DistrictName"].value_counts(dropna=False).head(10))

DistrictName
NaN                     83283
Los Angeles Unified     32314
San Diego Unified        8898
Desert Sands Unified     6084
Capistrano Unified       6027
Palm Springs Unified     5181
Oakland Unified          4291
Corona-Norco Unified     4254
Hemet Unified            4034
Long Beach Unified       3941
Name: count, dtype: int64


In [35]:
# Step 7: save the enriched dataset. This supersedes
# cleaned_single_family_sales_preprocessed.csv and is what the modeling
# section below — and every subsequent week — should read from.
df.to_csv("cleaned_single_family_sales_engineered.csv", index=False)
print(f"Saved cleaned_single_family_sales_engineered.csv — shape {df.shape}")

Saved cleaned_single_family_sales_engineered.csv — shape (333028, 29)


In [36]:
# =============================================================================
# Modeling: old vs. new feature sets, on the SAME enriched dataset/rows
# =============================================================================
TARGET = "ClosePrice"
LOW_CARD_CAT = [
    "PropertySubType", "Levels", "AttachedGarageYN",
    "PoolPrivateYN", "FireplaceYN", "NewConstructionYN",
]

In [37]:
# Re-normalize categorical dtypes defensively (same rationale as Week 4/5).
for col in ["City", "DistrictName"] + LOW_CARD_CAT:
    df[col] = df[col].where(df[col].isna(), df[col].astype(str))

In [38]:
BASE_NUMERIC_COLS = BASE_NUMERIC_BASE_NAMES + [
    f"{c}_missing" for c in BASE_NUMERIC_BASE_NAMES if f"{c}_missing" in df.columns
]
ENGINEERED_NUMERIC_COLS = ENGINEERED_BASE_NAMES + [
    f"{c}_missing" for c in ENGINEERED_BASE_NAMES if f"{c}_missing" in df.columns
]

In [39]:
print("Base numeric columns:", BASE_NUMERIC_COLS)
print("Engineered numeric columns:", ENGINEERED_NUMERIC_COLS)

Base numeric columns: ['LivingArea', 'Bedrooms', 'Bathrooms', 'LotSize', 'GarageSpaces', 'YearBuilt', 'ParkingTotal', 'LivingArea_missing', 'Bathrooms_missing', 'LotSize_missing', 'GarageSpaces_missing', 'YearBuilt_missing', 'ParkingTotal_missing']
Engineered numeric columns: ['BedBathRatio', 'PropertyAge', 'BedBathRatio_missing', 'PropertyAge_missing']


In [40]:
def month_period(series):
    return series.dt.to_period("M")

In [41]:
def get_window(data, end_month_exclusive, n_months):
    start = end_month_exclusive - n_months
    periods = month_period(data["CloseDate"])
    return data[(periods >= start) & (periods < end_month_exclusive)]

In [42]:
def get_month(data, month):
    return data[month_period(data["CloseDate"]) == month]

In [43]:
def kfold_target_encode(train_series, train_target, apply_series_list,
                         n_splits=5, smoothing=20, seed=RANDOM_SEED):
    """CV-safe target encoding (Sec. 05) — unchanged from Week 4/5. Used for
    both City and, now, DistrictName."""
    train_series = train_series.reset_index(drop=True)
    train_target = train_target.reset_index(drop=True)
    global_mean = train_target.mean()

    encoded_train = np.zeros(len(train_series))
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    for fit_idx, hold_idx in kf.split(train_series):
        fit_vals, fit_target = train_series.iloc[fit_idx], train_target.iloc[fit_idx]
        stats = fit_target.groupby(fit_vals).agg(["mean", "count"])
        smoothed = (stats["mean"] * stats["count"] + global_mean * smoothing) / (stats["count"] + smoothing)
        encoded_train[hold_idx] = train_series.iloc[hold_idx].map(smoothed).fillna(global_mean).to_numpy()

    full_stats = train_target.groupby(train_series).agg(["mean", "count"])
    full_smoothed = (full_stats["mean"] * full_stats["count"] + global_mean * smoothing) / (full_stats["count"] + smoothing)
    encoded_apply = [s.map(full_smoothed).fillna(global_mean).to_numpy() for s in apply_series_list]
    return encoded_train, encoded_apply

In [44]:
def compute_metrics(y_true, y_pred):
    return {"R2": r2_score(y_true, y_pred)}

In [45]:
def build_pipeline(regressor, numeric_final_cols, cat_final_cols):
    """Leak-safe ColumnTransformer (Sec. 07), generalized to accept whichever
    numeric/categorical column lists the current feature-set config needs.
    Only the final regressor step and the column lists change between runs;
    the imputer/scaler/encoder logic itself never does."""
    preprocessor = ColumnTransformer([
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]), numeric_final_cols),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", drop="first", min_frequency=10)),
        ]), cat_final_cols),
    ])
    return Pipeline([("preprocess", preprocessor), ("regressor", regressor)])

In [46]:
def fit_and_evaluate(train_df, eval_df, regressor, numeric_cols, low_card_cat, high_card_cat):
    """Generalized version of the Week 5 routine: which columns count as
    numeric / low-card categorical / high-card (target-encoded) categorical
    is now a parameter, so the same function drives every feature-set x
    model combination below.
    """
    train_df = train_df.copy()
    eval_df = eval_df.copy()

    # Outlier thresholds learned from TRAINING data only (Sec. 03).
    lo, hi = train_df[TARGET].quantile([0.005, 0.995])
    train_df = train_df[(train_df[TARGET] >= lo) & (train_df[TARGET] <= hi)]
    eval_df = eval_df[(eval_df[TARGET] >= lo) & (eval_df[TARGET] <= hi)]

    y_train = train_df[TARGET].reset_index(drop=True)
    y_eval = eval_df[TARGET].reset_index(drop=True)

    X_train = train_df.drop(columns=[TARGET, "CloseDate"]).reset_index(drop=True)
    X_eval = eval_df.drop(columns=[TARGET, "CloseDate"]).reset_index(drop=True)

    enc_cols = []
    for col in high_card_cat:
        enc_train, (enc_eval,) = kfold_target_encode(X_train[col], y_train, [X_eval[col]])
        enc_name = f"{col}_target_enc"
        X_train[enc_name] = enc_train
        X_eval[enc_name] = enc_eval
        enc_cols.append(enc_name)

    numeric_final = numeric_cols + enc_cols
    model = build_pipeline(regressor, numeric_final, low_card_cat)

    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", message="Found unknown categories.*", category=UserWarning)
        model.fit(X_train, y_train)
        y_pred_eval = model.predict(X_eval)
        y_pred_train = model.predict(X_train)

    eval_metrics = compute_metrics(y_eval, y_pred_eval)
    train_metrics = compute_metrics(y_train, y_pred_train)
    return model, train_metrics, eval_metrics, y_eval, y_pred_eval

In [47]:
# ---------------------------------------------------------------------------
# Chronological split (Sec. 01) — recomputed on the enriched dataset, since
# the new cleaning steps (missing coords, YearBuilt-after-CloseDate) can
# shift which months qualify or how many rows they contain.
# ---------------------------------------------------------------------------
months_sorted = sorted(month_period(df["CloseDate"]).unique())
if len(months_sorted) < 3:
    raise ValueError("Need at least 3 distinct months for a train/validation/test split.")

In [48]:
test_month = months_sorted[-1]
val_month = months_sorted[-2]
print(f"Validation month: {val_month}   Test month: {test_month}")

Validation month: 2026-05   Test month: 2026-06


In [49]:
def make_models():
    return {
        "Linear Regression": LinearRegression(),
        "Decision Tree": DecisionTreeRegressor(max_depth=10, min_samples_leaf=20, random_state=RANDOM_SEED),
        "Random Forest": RandomForestRegressor(
            n_estimators=300, max_depth=14, min_samples_leaf=5, n_jobs=-1, random_state=RANDOM_SEED,
        ),
    }

In [50]:
# Three feature-set configurations, isolating the district layer's specific
# contribution from the two engineered numeric features.
FEATURE_SETS = {
    "Old (Week 5)": {
        "numeric": BASE_NUMERIC_COLS,
        "low_card": LOW_CARD_CAT,
        "high_card": ["City"],
    },
    "New: engineered numeric": {
        "numeric": BASE_NUMERIC_COLS + ENGINEERED_NUMERIC_COLS,
        "low_card": LOW_CARD_CAT,
        "high_card": ["City"],
    },
    "New: + school district": {
        "numeric": BASE_NUMERIC_COLS + ENGINEERED_NUMERIC_COLS,
        "low_card": LOW_CARD_CAT,
        "high_card": ["City", "DistrictName"],
    },
}

In [51]:
candidate_X = [3, 6, 9, 12, 18, 24]

In [52]:
# ---------------------------------------------------------------------------
# Tune X per (feature set, model) on the validation month only (Sec. 01).
# ---------------------------------------------------------------------------
best_X = {}

In [53]:
for fs_name, fs_cfg in FEATURE_SETS.items():
    for model_name, regressor in make_models().items():
        key = (fs_name, model_name)
        print(f"\nTuning X — feature set: {fs_name} | model: {model_name}")
        val_results = {}
        for x in candidate_X:
            train_window = get_window(df, val_month, x)
            val_set = get_month(df, val_month)
            if len(train_window) < 100 or len(val_set) < 20:
                continue
            _, _, val_metrics, _, _ = fit_and_evaluate(
                train_window, val_set, regressor,
                fs_cfg["numeric"], fs_cfg["low_card"], fs_cfg["high_card"],
            )
            val_results[x] = val_metrics["R2"]
            print(f"  X={x:>2} months -> val R2={val_metrics['R2']:.4f}")
        if not val_results:
            raise ValueError(f"No candidate training window had enough data for {key}.")
        best_X[key] = max(val_results, key=val_results.get)
        print(f"  Selected X = {best_X[key]} months")


Tuning X — feature set: Old (Week 5) | model: Linear Regression
  X= 3 months -> val R2=0.7474
  X= 6 months -> val R2=0.7561
  X= 9 months -> val R2=0.7642
  X=12 months -> val R2=0.7649
  X=18 months -> val R2=0.7662
  X=24 months -> val R2=0.7670
  Selected X = 24 months

Tuning X — feature set: Old (Week 5) | model: Decision Tree
  X= 3 months -> val R2=0.7692
  X= 6 months -> val R2=0.7866
  X= 9 months -> val R2=0.8026
  X=12 months -> val R2=0.8070
  X=18 months -> val R2=0.8027
  X=24 months -> val R2=0.8082
  Selected X = 24 months

Tuning X — feature set: Old (Week 5) | model: Random Forest
  X= 3 months -> val R2=0.8077
  X= 6 months -> val R2=0.8217
  X= 9 months -> val R2=0.8352
  X=12 months -> val R2=0.8374
  X=18 months -> val R2=0.8389
  X=24 months -> val R2=0.8392
  Selected X = 24 months

Tuning X — feature set: New: engineered numeric | model: Linear Regression
  X= 3 months -> val R2=0.7508
  X= 6 months -> val R2=0.7596
  X= 9 months -> val R2=0.7682
  X=12 mont

In [54]:
# ---------------------------------------------------------------------------
# Final comparison: retrain each (feature set, model) with its own selected
# X, evaluate once on the test month.
# ---------------------------------------------------------------------------
rows = []
fitted_models = {}

In [55]:
for fs_name, fs_cfg in FEATURE_SETS.items():
    for model_name, regressor in make_models().items():
        x = best_X[(fs_name, model_name)]
        final_train = get_window(df, test_month, x)
        test_set = get_month(df, test_month)
        model, train_metrics, test_metrics, y_eval, y_pred = fit_and_evaluate(
            final_train, test_set, regressor,
            fs_cfg["numeric"], fs_cfg["low_card"], fs_cfg["high_card"],
        )
        fitted_models[(fs_name, model_name)] = model
        rows.append({
            "Feature Set": fs_name,
            "Model": model_name,
            "Training window X": x,
            "Train R2": round(train_metrics["R2"], 4),
            "Test R2": round(test_metrics["R2"], 4),
        })

In [56]:
comparison_long = pd.DataFrame(rows)

In [57]:
print("=== Old vs. New Feature Sets — Test R2 by model (long form) ===\n")
print(comparison_long.to_string(index=False))

=== Old vs. New Feature Sets — Test R2 by model (long form) ===

            Feature Set             Model  Training window X  Train R2  Test R2
           Old (Week 5) Linear Regression                 24    0.7664   0.7701
           Old (Week 5)     Decision Tree                 24    0.8352   0.8131
           Old (Week 5)     Random Forest                 24    0.8939   0.8419
New: engineered numeric Linear Regression                 24    0.7676   0.7723
New: engineered numeric     Decision Tree                 24    0.8357   0.8169
New: engineered numeric     Random Forest                 24    0.8952   0.8446
 New: + school district Linear Regression                 24    0.7682   0.7729
 New: + school district     Decision Tree                 24    0.8387   0.8195
 New: + school district     Random Forest                 24    0.9002   0.8512


In [58]:
print("\n=== Same data, pivoted: Test R2 by Model x Feature Set ===\n")
pivot = comparison_long.pivot(index="Model", columns="Feature Set", values="Test R2")
pivot = pivot[["Old (Week 5)", "New: engineered numeric", "New: + school district"]]
print(pivot.to_string())


=== Same data, pivoted: Test R2 by Model x Feature Set ===

Feature Set        Old (Week 5)  New: engineered numeric  New: + school district
Model                                                                           
Decision Tree            0.8131                   0.8169                  0.8195
Linear Regression        0.7701                   0.7723                  0.7729
Random Forest            0.8419                   0.8446                  0.8512


In [59]:
# ---------------------------------------------------------------------------
# Isolate the incremental value of each addition, per model.
# ---------------------------------------------------------------------------
print("=== Incremental R2 gain per addition, by model ===\n")
for model_name in pivot.index:
    old = pivot.loc[model_name, "Old (Week 5)"]
    eng = pivot.loc[model_name, "New: engineered numeric"]
    dist = pivot.loc[model_name, "New: + school district"]
    print(f"{model_name}:")
    print(f"  BedBathRatio + PropertyAge:   {eng - old:+.4f}")
    print(f"  + School district (Unified):  {dist - eng:+.4f}")
    print(f"  Total vs. Week 5 baseline:     {dist - old:+.4f}\n")

=== Incremental R2 gain per addition, by model ===

Decision Tree:
  BedBathRatio + PropertyAge:   +0.0038
  + School district (Unified):  +0.0026
  Total vs. Week 5 baseline:     +0.0064

Linear Regression:
  BedBathRatio + PropertyAge:   +0.0022
  + School district (Unified):  +0.0006
  Total vs. Week 5 baseline:     +0.0028

Random Forest:
  BedBathRatio + PropertyAge:   +0.0027
  + School district (Unified):  +0.0066
  Total vs. Week 5 baseline:     +0.0093



In [60]:
# ---------------------------------------------------------------------------
# Feature importances under the full "+ school district" feature set, for
# the two tree-based models — shows where DistrictName lands relative to
# City and the other engineered features.
# ---------------------------------------------------------------------------
for model_name in ("Decision Tree", "Random Forest"):
    model = fitted_models[("New: + school district", model_name)]
    feature_names = model.named_steps["preprocess"].get_feature_names_out()
    importances = model.named_steps["regressor"].feature_importances_
    top = pd.Series(importances, index=feature_names).sort_values(ascending=False).head(10)
    print(f"\nTop 10 features — {model_name} (+ school district):")
    print(top.to_string())


Top 10 features — Decision Tree (+ school district):
num__City_target_enc            0.628006
num__LivingArea                 0.329350
num__DistrictName_target_enc    0.008332
num__YearBuilt                  0.007438
cat__PoolPrivateYN_Unknown      0.007168
num__LotSize                    0.005313
num__BedBathRatio               0.005046
num__PropertyAge                0.002881
num__Bathrooms                  0.001948
cat__Levels_Unknown             0.001026

Top 10 features — Random Forest (+ school district):
num__City_target_enc            0.595809
num__LivingArea                 0.316835
num__LotSize                    0.017809
num__DistrictName_target_enc    0.015111
num__YearBuilt                  0.011778
num__PropertyAge                0.009520
num__BedBathRatio               0.007488
cat__PoolPrivateYN_Unknown      0.006945
num__Bathrooms                  0.003480
num__ParkingTotal               0.003039


In [61]:
# ---------------------------------------------------------------------------
# Save comparison table
# ---------------------------------------------------------------------------
comparison_long.to_csv("feature_set_comparison.csv", index=False)
print("Saved feature_set_comparison.csv")

Saved feature_set_comparison.csv
